### CARREGAR BASES

In [0]:
import os
from pyspark.sql import SparkSession

# Caminho da pasta onde estão os arquivos
folder_path = '/Workspace/Users/XXXXXXX@gmail.com/POSTECH - DTAT - Tech Challenge - Fase 1/'

# Inicializando a sessão Spark (opcional se já existir)
spark = SparkSession.builder.getOrCreate()

dfs = {}

# Lista todos os arquivos na pasta
table_files = [f for f in os.listdir(folder_path) if f.endswith('.csv') or f.endswith('.parquet')]

# Mapear nomes dos arquivos para nomes de DataFrames e descrições
nome_map = {
    'olist_customers_dataset': ('clientes', 'Clientes: Identificadores e localização dos clientes'),
    'olist_orders_dataset': ('pedidos', 'Pedidos: Status, compra, aprovação e entrega dos pedidos'),
    'olist_order_items_dataset': ('itens_pedidos', 'Itens dos pedidos: Itens, produtos, vendedores, preços e fretes'),
    'olist_order_payments_dataset': ('pagamentos', 'Pagamentos: Modalidades, parcelas e valores de pagamento'),
    'olist_order_reviews_dataset': ('avaliacoes', 'Avaliações: Notas, comentários e datas das avaliações'),
    'olist_products_dataset': ('produtos', 'Produtos: Categorias, dimensões, peso e atributos dos produtos'),
    'olist_sellers_dataset': ('vendedores', 'Vendedores: Identificadores e localização dos vendedores'),
    'olist_geolocation_dataset': ('geolocalizacao', 'Geolocalização: Prefixos de CEP e coordenadas geográficas'),
    'product_category_name_translation': ('traducao_categorias', 'Tradução categorias: Tradução das categorias de português para inglês'),
}

# Carregar os DataFrames e atribuir aos nomes específicos
for table_file in table_files:
    file_path = os.path.join(folder_path, table_file)
    table_name = os.path.splitext(table_file)[0]
    if table_name not in nome_map:
        continue
    df_name, descricao = nome_map[table_name]
    if table_file.endswith('.csv'):
        df = spark.read.csv(file_path, header=True, inferSchema=True)
    elif table_file.endswith('.parquet'):
        df = spark.read.parquet(file_path)
    else:
        continue
    globals()[df_name] = df  # Cria variável com nome específico para o DataFrame
    print(descricao)
    print(table_name)
    display(df)


In [0]:
# Verificar valores ausentes em todos os DataFrames carregados
import pyspark
for nome, df in list(globals().items()):
    from pyspark.sql import functions as F
    if isinstance(df, pyspark.sql.dataframe.DataFrame):
        print(f"\nDataFrame: {nome}")
        for col in df.columns:
            n_ausentes = df.filter(F.col(col).isNull()).count()
            print(f"   Coluna '{col}': {n_ausentes} valores ausentes")


### EXPLORAÇÃO DOS DADOS

In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 1. Evolução de Vendas 

from pyspark.sql.functions import col, concat_ws, year, month, expr

# Adiciona coluna "ano_mes" com ano e mês juntos
pedidos_com_mes = pedidos.withColumn(
    "ano_mes",
    concat_ws("-", year(col("order_purchase_timestamp")), month(col("order_purchase_timestamp")))
)

# Conta pedidos distintos por ano-mes
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, format_string

pedidos_por_mes = pedidos_com_mes.groupBy("ano_mes").agg(
    expr("count(distinct order_id) as n_pedidos")
).orderBy("ano_mes")

# Calcula percentual em relação ao mês anterior
window_spec = Window.orderBy("ano_mes")
pedidos_por_mes = pedidos_por_mes.withColumn(
    "n_pedidos_anterior",
    lag("n_pedidos").over(window_spec)
)
pedidos_por_mes = pedidos_por_mes.withColumn(
    "percentual_vs_mes_anterior",
    expr("ROUND(100.0 * (n_pedidos - n_pedidos_anterior) / n_pedidos_anterior, 2)")
)

# Adiciona o total de pedidos no período
from pyspark.sql.functions import sum as spark_sum

total_pedidos = pedidos_por_mes.agg(spark_sum('n_pedidos').alias('total_pedidos')).collect()[0]['total_pedidos']

print(f'Total de pedidos no período: {total_pedidos}')

# Exibe resultado
display(pedidos_por_mes)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 2. Receita Bruta

from pyspark.sql.functions import year, month, concat_ws, sum as spark_sum

# Adiciona coluna "ano_mes" com ano e mês juntos
itens_pedidos_com_mes = itens_pedidos.withColumn(
    "ano_mes",
    concat_ws("-", year(itens_pedidos["shipping_limit_date"]), month(itens_pedidos["shipping_limit_date"]))
)

# Receita bruta mensal (soma dos preços dos itens)
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, format_string, col

receita_mensal = itens_pedidos_com_mes.groupBy("ano_mes").agg(
    spark_sum("price").alias("receita_bruta")
).orderBy("ano_mes")

# Calcula o percentual em relação ao mês anterior
window_spec = Window.orderBy("ano_mes")
receita_mensal = receita_mensal.withColumn(
    "receita_anterior",
    lag("receita_bruta").over(window_spec)
)
receita_mensal = receita_mensal.withColumn(
    "percentual_vs_mes_anterior",
    (col("receita_bruta") / col("receita_anterior") * 100 - 100)
)

display(receita_mensal)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 3. Pedidos por Região 

# Cruzar pedidos e clientes pelo customer_id
pedidos_clientes = pedidos.join(clientes, pedidos.customer_id == clientes.customer_id, 'inner')

# Cruzar com geolocalizacao via zip_code_prefix
pedidos_clientes_geo = pedidos_clientes.join(
    geolocalizacao, pedidos_clientes.customer_zip_code_prefix == geolocalizacao.geolocation_zip_code_prefix, 'left')

# Exibir base consolidada
# Remover linhas duplicadas
pedidos_clientes_geo_distintos = pedidos_clientes_geo.dropDuplicates(['order_id'])

# Contar order_id distintos
total_order_id_distintos = pedidos_clientes_geo_distintos.select('order_id').distinct().count()
print(f"Total de order_id distintos na base consolidada: {total_order_id_distintos}")
display(pedidos_clientes_geo_distintos)


# Contar order_id distintos agrupados por customer_state
from pyspark.sql.functions import countDistinct

from pyspark.sql.functions import year, month, concat_ws

# Adiciona coluna 'mes_ano' (ano-mês)
pedidos_clientes_geo_distintos = pedidos_clientes_geo_distintos.withColumn(
    'mes_ano', concat_ws("-", year('order_purchase_timestamp'), month('order_purchase_timestamp'))
)

# Distribuição mensal por região (ano-mês na mesma coluna, valores por região, ordena do maior p/ menor mes_ano)
dist_mensal_regiao = pedidos_clientes_geo_distintos.groupBy('mes_ano', 'customer_state').agg(
    countDistinct('order_id').alias('n_pedidos')
).orderBy('mes_ano', ascending=False)
display(dist_mensal_regiao)

# Distribuição por ano por região (ano na mesma coluna, valores por região, ordena do maior p/ menor ano)
pedidos_clientes_geo_distintos = pedidos_clientes_geo_distintos.withColumn('ano', year('order_purchase_timestamp'))
dist_ano_regiao = pedidos_clientes_geo_distintos.groupBy('ano', 'customer_state').agg(
    countDistinct('order_id').alias('n_pedidos')
).orderBy('ano', ascending=False)
display(dist_ano_regiao)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 4. Visão por categoria

from pyspark.sql.functions import year, month, concat_ws, sum as spark_sum, countDistinct

# Adiciona coluna "ano_mes" com ano e mês juntos (usando shipping_limit_date, se desejado pode usar outra data)
pedidos_itens_produtos_final_mes = pedidos_itens_produtos_final.withColumn(
    "ano_mes",
    concat_ws("-", year(pedidos_itens_produtos_final["shipping_limit_date"]), month(pedidos_itens_produtos_final["shipping_limit_date"]))
)

# Visão mensal: soma de price, número de pedidos distintos (sem duplicar order_id) e média de price por pedido
from pyspark.sql.functions import avg

# Identifica orders com mais de uma categoria
multi_cat_orders = pedidos_itens_produtos_final_mes.groupBy("order_id") \
    .agg(countDistinct("product_category_name").alias("num_categorias")) \
    .filter("num_categorias > 1")

# Recupera todas as linhas dos pedidos com >1 categoria
pedidos_multicategoria = pedidos_itens_produtos_final_mes.join(multi_cat_orders.select("order_id"), on="order_id", how="inner")

display(pedidos_multicategoria)

# Visão mensal: soma de price, número de pedidos distintos (sem duplicar order_id) e média de price por pedido
visao_mensal = pedidos_itens_produtos_final_mes.dropDuplicates(["order_id"]).groupBy("product_category_name", "ano_mes").agg(
    spark_sum("price").alias("soma_price_mensal"),
    countDistinct("order_id").alias("num_order_id_distintos"),
    (spark_sum("price")/countDistinct("order_id")).alias("media_price_por_pedido")
).orderBy(["ano_mes", "product_category_name"])

display(visao_mensal)

# Soma price agrupado por categoria de produto e mês-ano
soma_price_categoria_mes = pedidos_itens_produtos_final_mes.groupBy("product_category_name", "ano_mes").agg(
    spark_sum("price").alias("soma_price")
).orderBy(["ano_mes", "soma_price"], ascending=[True, False])

from pyspark.sql.functions import coalesce, lit

df_order_id_distintos = pedidos_itens_produtos_final.withColumn(
    "product_category_name",
    coalesce(pedidos_itens_produtos_final["product_category_name"], lit("Não Informada"))
).groupBy("product_category_name").agg(
    countDistinct("order_id").alias("num_order_id_distintos")
).orderBy("num_order_id_distintos", ascending=False)




In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 5. Leadtime - Gargalos Etapas

from pyspark.sql.functions import col, year, month, concat_ws, datediff

# Adiciona coluna "ano_mes" com ano e mês juntos (da compra)
pedidos_leadtime = pedidos.withColumn(
    "ano_mes",
    concat_ws("-", year(col("order_purchase_timestamp")), month(col("order_purchase_timestamp")))
)

# Lead time entre etapas
pedidos_leadtime = pedidos_leadtime.withColumn(
    "leadtime_compra_aprovacao", 
    datediff(col("order_approved_at"), col("order_purchase_timestamp"))
).withColumn(
    "leadtime_aprovacao_envio", 
    datediff(col("order_delivered_carrier_date"), col("order_approved_at"))
).withColumn(
    "leadtime_envio_entrega", 
    datediff(col("order_delivered_customer_date"), col("order_delivered_carrier_date"))
).withColumn(
    "leadtime_total_previsto", 
    datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp"))
).withColumn(
    "leadtime_total_real", 
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
)

# Comparativos entre etapas para identificar gargalos
from pyspark.sql.functions import avg, expr

# Calcula médias mensais das etapas
from pyspark.sql.functions import year

# Adiciona coluna "ano" extraída de order_purchase_timestamp
pedidos_leadtime = pedidos_leadtime.withColumn(
    "ano", year(col("order_purchase_timestamp"))
)

leadtime_anual = pedidos_leadtime.groupBy("ano").agg(
    avg("leadtime_compra_aprovacao").alias("media_compra_aprovacao"),
    avg("leadtime_aprovacao_envio").alias("media_aprovacao_envio"),
    avg("leadtime_envio_entrega").alias("media_envio_entrega"),
    avg("leadtime_total_previsto").alias("media_total_previsto"),
    (
        avg("leadtime_compra_aprovacao") + avg("leadtime_aprovacao_envio") + avg("leadtime_envio_entrega")
    ).alias("media_total_real")
).orderBy("ano")

# Calcula percentual de cada etapa em relação ao lead time total real
def calc_percent(col_etapa, col_total):
    return expr(f"ROUND(({{col_etapa}} / NULLIF({{col_total}},0))*100,2)".format(col_etapa=col_etapa, col_total=col_total))

leadtime_anual_percent = leadtime_anual\
    .withColumn("pct_compra_aprovacao_real", calc_percent("media_compra_aprovacao", "media_total_real"))\
    .withColumn("pct_aprovacao_envio_real", calc_percent("media_aprovacao_envio", "media_total_real"))\
    .withColumn("pct_envio_entrega_real", calc_percent("media_envio_entrega", "media_total_real"))


display(leadtime_anual_percent)

# Consolidado mensal (mantém original)
leadtime_mensal = pedidos_leadtime.groupBy("ano_mes").agg(
    avg("leadtime_compra_aprovacao").alias("media_compra_aprovacao"),
    avg("leadtime_aprovacao_envio").alias("media_aprovacao_envio"),
    avg("leadtime_envio_entrega").alias("media_envio_entrega"),
    avg("leadtime_total_previsto").alias("media_total_previsto"),
    (
        avg("leadtime_compra_aprovacao") + avg("leadtime_aprovacao_envio") + avg("leadtime_envio_entrega")
    ).alias("media_total_real")
).orderBy("ano_mes")

# Adiciona percentuais das etapas em relação ao total real
leadtime_mensal_percent = leadtime_mensal\
    .withColumn("pct_compra_aprovacao_real", calc_percent("media_compra_aprovacao", "media_total_real"))\
    .withColumn("pct_aprovacao_envio_real", calc_percent("media_aprovacao_envio", "media_total_real"))\
    .withColumn("pct_envio_entrega_real", calc_percent("media_envio_entrega", "media_total_real"))

# Exibe DataFrame comparativo
display(leadtime_mensal_percent)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 6. Leadtime - Médias por região

from pyspark.sql.functions import col, year, month, concat_ws, datediff

# Adiciona coluna "ano_mes" com ano e mês juntos (da compra)
pedidos_leadtime = pedidos.withColumn(
    "ano_mes",
    concat_ws("-", year(col("order_purchase_timestamp")), month(col("order_purchase_timestamp")))
)

# Lead time entre compra e entrega prevista/real
def_padrao = pedidos_leadtime.withColumn(
    "leadtime_compra_entrega_previsto",
    datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp"))
).withColumn(
    "leadtime_compra_entrega_real",
    datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
)

# Join com clientes pelo customer_id
def_padrao_cliente = def_padrao.join(clientes, on="customer_id", how="left")

# Depois cruzar com geolocalizacao por customer_zip_code_prefix = geolocation_zip_code_prefix
def_padrao_geo = def_padrao_cliente.join(
    geolocalizacao, 
    def_padrao_cliente.customer_zip_code_prefix == geolocalizacao.geolocation_zip_code_prefix, 
    how="left"
)

# Exibir resultado final
df_final = def_padrao_geo.drop('geolocation_lat', 'geolocation_city').dropDuplicates()

display(df_final)

from pyspark.sql.functions import avg, year, month, concat_ws

# Adicionar coluna "ano_mes" à base consolidada
media_leadtime_df = df_final.withColumn(
    "ano_mes",
    concat_ws("-", year(df_final["order_purchase_timestamp"]), month(df_final["order_purchase_timestamp"]))
)

# Visão mensal por estado
media_leadtime_por_mes_estado = media_leadtime_df.groupBy("ano_mes", "customer_state").agg(
    avg("leadtime_compra_entrega_previsto").alias("media_leadtime_previsto"),
    avg("leadtime_compra_entrega_real").alias("media_leadtime_real")
)
media_leadtime_por_mes_estado = media_leadtime_por_mes_estado.withColumn(
    "diferenca_real_previsto",
    media_leadtime_por_mes_estado["media_leadtime_real"] - media_leadtime_por_mes_estado["media_leadtime_previsto"]
)

# Exibir resultado
print("Visão mensal por estado:")
display(media_leadtime_por_mes_estado)

# Visão apenas por estado (sem mês)
media_leadtime_por_estado = df_final.groupBy("customer_state").agg(
    avg("leadtime_compra_entrega_previsto").alias("media_leadtime_previsto"),
    avg("leadtime_compra_entrega_real").alias("media_leadtime_real")
)
media_leadtime_por_estado = media_leadtime_por_estado.withColumn(
    "diferenca_real_previsto",
    media_leadtime_por_estado["media_leadtime_real"] - media_leadtime_por_estado["media_leadtime_previsto"]
)

print("Visão total por estado:")
display(media_leadtime_por_estado)

In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 7. Prazo Entrega vs NPS - Geral

from pyspark.sql.functions import col, when, coalesce, lit

# Juntar os DataFrames de pedidos e avaliações pelo order_id
df_avaliacoes = pedidos.join(avaliacoes, on="order_id", how="left")

# Criar coluna de status de entrega (no prazo/atraso)
df_avaliacoes = df_avaliacoes.withColumn(
    "status_entrega",
    when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), "Entrega com atraso")
    .otherwise("Entrega no prazo")
)

from pyspark.sql.functions import count, year, month, concat_ws, sum as sum_

# Adiciona coluna ano_mes para período
df_avaliacoes = df_avaliacoes.withColumn(
    "ano_mes",
    concat_ws("-", year(col("order_estimated_delivery_date")), month(col("order_estimated_delivery_date")))
)

# Categoriza review_score nulo como "Não avaliado"
df_avaliacoes = df_avaliacoes.withColumn(
    "review_score_cat",
    coalesce(col("review_score"), lit("Não avaliado"))
)

# Agrupa por status_entrega, ano_mes e review_score_cat, conta recorrência
from pyspark.sql import Window
from pyspark.sql.functions import round

recorrencia_review_score = df_avaliacoes.groupBy(
    "status_entrega", "ano_mes", "review_score_cat"
).agg(count("order_id").alias("recorrencia"))

# Calcular o percentual de review_score_cat por recorrencia para cada grupo status_entrega dentro do mes
window_spec = Window.partitionBy("ano_mes", "status_entrega")
recorrencia_review_score = recorrencia_review_score.withColumn(
    "percentual_review_score_cat",
    round(col("recorrencia") / (sum_(col("recorrencia")).over(window_spec)) * 100, 2)
)

# Exibir resultado
display(recorrencia_review_score.orderBy(["ano_mes", "status_entrega", "review_score_cat"]))


In [0]:
# CÓDIGO DE CRUZAMENTO DE BASE PARA ANÁLISES SEGUINTES COMPLEMENTADO COM BASE SE CLIENTES E GEOLOCALIZAÇÃO

# Cruzar Clientes, Pedidos, Geolocalização e Avaliações
from pyspark.sql.functions import col, when

def cross_tables(clientes, pedidos, geolocalizacao, avaliacoes):
    # Join clientes e pedidos por customer_id
    clientes_pedidos = clientes.join(pedidos, on="customer_id", how="inner")

    # Join com geolocalizacao por zip_code_prefix
    clientes_pedidos_geo = clientes_pedidos.join(
        geolocalizacao,
        clientes_pedidos.customer_zip_code_prefix == geolocalizacao.geolocation_zip_code_prefix,
        how="left"
    )

    # Join com avaliacoes por order_id
    clientes_pedidos_geo_avaliacoes = clientes_pedidos_geo.join(
        avaliacoes,
        on="order_id",
        how="left"
    )

    # Criar coluna de status de entrega (no prazo/atraso)
    clientes_pedidos_geo_avaliacoes = clientes_pedidos_geo_avaliacoes.withColumn(
        "status_entrega",
        when(
            col("order_delivered_customer_date") > col("order_estimated_delivery_date"),
            "Entrega com atraso"
        ).otherwise("Entrega no prazo")
    )

    # Selecionar apenas colunas únicas (sem duplicação)
    cols = []
    seen = set()
    for c in clientes_pedidos_geo_avaliacoes.columns:
        if c not in seen:
            cols.append(c)
            seen.add(c)

    # Remove linhas duplicadas considerando o order_id
    resultado_final = clientes_pedidos_geo_avaliacoes.select(*cols).dropDuplicates(["order_id"])
    return resultado_final

resultado = cross_tables(clientes, pedidos, geolocalizacao, avaliacoes)
display(resultado)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 8. Prazo Entrega vs NPS - Det

from pyspark.sql.functions import countDistinct, sum as sum_

# Contar order_id distintos por status_entrega e geolocation_state
from pyspark.sql import Window
from pyspark.sql.functions import col, round

contagem_status_estado = resultado.groupBy("status_entrega", "geolocation_state").agg(
    countDistinct("order_id").alias("num_order_id_distintos")
)

# Calcular o percentual de cada status_entrega dentro de cada geolocation_state
window_estado = Window.partitionBy("geolocation_state")
contagem_status_estado = contagem_status_estado.withColumn(
    "percentual_status_entrega_por_estado",
    round(col("num_order_id_distintos") / sum_(col("num_order_id_distintos")).over(window_estado) * 100, 2)
)

# Calcular o percentual de cada grupo status_entrega+estado em relação ao total geral
window_total = Window.partitionBy()
contagem_status_estado = contagem_status_estado.withColumn(
    "percentual_status_estado_total",
    round(col("num_order_id_distintos") / sum_(col("num_order_id_distintos")).over(window_total) * 100, 2)
)

display(contagem_status_estado)

# Contar order_id distintos por status_entrega, review_score e geolocation_state
contagem_status_review_estado = resultado.groupBy("status_entrega", "review_score", "geolocation_state").agg(
    countDistinct("order_id").alias("num_order_id_distintos")
)
display(contagem_status_review_estado)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 9. Média de compra por cliente

from pyspark.sql.functions import countDistinct, avg

# Exibir resultado e agrupar por status_entrega e review_score, com média de compras por cliente único
# customer_unique_id é o id único do cliente

grupo = resultado.groupBy("status_entrega", "review_score").agg(
    countDistinct("customer_unique_id").alias("num_clientes_unicos"),
    countDistinct("order_id").alias("num_compras"),
    (countDistinct("order_id") / countDistinct("customer_unique_id")).alias("media_compras_por_cliente")
)

display(grupo)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 10. Análise NPS Baixos e 10.1 Amostras categoria Outros

from pyspark.sql.functions import col

# Filtrar notas 1, 2, 3
resultados_baixas = resultado.filter(col("review_score").isin([1, 2, 3]))

# Exibir exemplos dos comentários (resumo das reclamações)
display(resultados_baixas.select("status_entrega", "review_score", "review_comment_message").where(col("review_comment_message").isNotNull()))

# Para resumo de reclamação, gerar amostra de frases mais comuns (exibição por wordcloud ou value_counts pode ser feita externamente)
from pyspark.sql.functions import lower, regexp_replace, split, explode, count
# Categorizar motivo das reclamações com base em palavras-chave comuns
from pyspark.sql.functions import when

motivos = resultados_baixas.withColumn(
    "motivo",
    when(col("review_comment_message").rlike(r"atras|demora|entrega|tempo"), "Atraso/Entrega")
    .when(col("review_comment_message").rlike(r"produto|produto errado|defeito|danificad|quebrad"), "Produto (Defeito/Errado)")
    .when(col("review_comment_message").rlike(r"atendimento|suporte|maltratad|ignorad|descaso"), "Atendimento")
    .when(col("review_comment_message").rlike(r"cancel|reembols|devolu|restitui"), "Cancelamento/Reembolso")
    .when(col("review_comment_message").rlike(r"frete|cobranca|taxa|valor frete"), "Frete/Valor")
    .when(col("review_comment_message").rlike(r"qualidade|ruim|bom|péssimo|excelente|item avariado|item danificado|item quebrado|item errado"), "Qualidade/Problema Item")
    .otherwise("Outro")
)

# Ver a distribuição dos motivos
from pyspark.sql.functions import sum as spark_sum, round as spark_round

total_motivos = motivos.count()
from pyspark.sql.window import Window

frequencia_motivos = motivos.groupBy("motivo", "review_score").count().orderBy(["motivo", "review_score"], ascending=[False, False])

window_review = Window.partitionBy('review_score')
frequencia_motivos = frequencia_motivos.withColumn(
    "percentual",
    spark_round((col("count") / spark_sum(col("count")).over(window_review)) * 100, 2)
)
display(frequencia_motivos)

# Amostras de comentários por motivo e review_score
from pyspark.sql.functions import rand

for nota in [1, 2, 3]:
    amostras_outro = motivos.filter((col("review_score") == nota) & (col("motivo") == "Outro") & (col("review_comment_message").isNotNull())) \
        .select("status_entrega", "review_score", "review_comment_message") \
        .orderBy(rand()).limit(385)
    print(f"Amostras para nota {nota} (categoria 'Outro'):")
    display(amostras_outro)

# Exibir resumo por status_entrega e review_score
total_pedidos = resultados_baixas.count()
grupo_status_review = resultados_baixas\
    .groupBy("status_entrega", "review_score")\
    .agg(count("order_id").alias("num_pedidos"))\
    .orderBy(["status_entrega", "review_score"])
grupo_status_review = grupo_status_review.withColumn(
    "percentual",
    spark_round((col("num_pedidos") / total_pedidos) * 100, 2)
)
print("Resumo de pedidos por status_entrega e review_score:")
display(grupo_status_review)


In [0]:
# CÓDIGO PARA RESULTADO ABA DO EXCEL: 11. Média de Preço vs Frete

from pyspark.sql.functions import avg

from pyspark.sql.functions import avg, year, month, dayofmonth, concat_ws

# Calcular média de frete (freight_value) e média de valor de produto (price)
resumo_frete_produto = itens_pedidos.agg(
    avg("price").alias("media_price"),
    avg("freight_value").alias("media_freight")
)

# Visão mensal: adiciona coluna "ano_mes"
itens_pedidos_mes = itens_pedidos.withColumn("ano_mes", concat_ws("-", year("shipping_limit_date"), month("shipping_limit_date")))

resumo_mensal = itens_pedidos_mes.groupBy("ano_mes").agg(
    avg("price").alias("media_price"),
    avg("freight_value").alias("media_freight")
).orderBy("ano_mes")

# Visão diária: adiciona coluna "ano_mes_dia"
itens_pedidos_dia = itens_pedidos.withColumn("ano_mes_dia", concat_ws("-", year("shipping_limit_date"), month("shipping_limit_date"), dayofmonth("shipping_limit_date")))

resumo_diario = itens_pedidos_dia.groupBy("ano_mes_dia").agg(
    avg("price").alias("media_price"),
    avg("freight_value").alias("media_freight")
).orderBy("ano_mes_dia")

# Exibir resultado como DataFrame
import pandas as pd
resumo_frete_produto_pd = resumo_frete_produto.toPandas()
display(resumo_frete_produto_pd)
display(resumo_mensal)
display(resumo_diario)

# Relação entre média de frete e média de valor do produto
media_price = resumo_frete_produto_pd.loc[0, "media_price"]
media_freight = resumo_frete_produto_pd.loc[0, "media_freight"]
print(f"Média de valor de produto: R$ {media_price:.2f}")
print(f"Média de frete: R$ {media_freight:.2f}")
print(f"Relação frete/produto: {media_freight/media_price:.2%}")
